# Comparison bench — leaderboard across every `01-rag/` leaf

This leaf has no model of its own. It walks every sibling `eval-snapshot.json` and produces a single Markdown table for the README. The same script is also run from `eval.py` to regenerate the snapshot.

The headline metric per technique is picked by a small, explicit dispatch table — no fragile auto-discovery. When you add a new leaf, register its metric here.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import json
from pathlib import Path
from shared.paths import repo_root

ROOT = repo_root() / '01-rag'
HEADLINES = {
    'naive-rag':            ('metrics.context_recall',            'context_recall'),
    'chunking-strategies':  ('metrics.per_strategy.fixed.avg_chunks_per_doc', 'avg chunks/doc (fixed)'),
    'embedding-comparison': ('metrics.per_config.large-512.recall@3', 'recall@3 (large-512)'),
    'hybrid-search':        ('metrics.per_retriever.hybrid_rrf.recall@3',  'recall@3 (hybrid RRF)'),
    'reranking':            ('metrics.recall@1.llm',                       'recall@1 (LLM reranker)'),
    'query-transformation': ('metrics.per_strategy.hyde.recall@3',         'recall@3 (HyDE)'),
    'self-rag':             ('metrics.unanswerable_refusal_rate',          'refusal_rate (unanswerable)'),
    'corrective-rag':       ('metrics.verdict_accuracy',                   'verdict_accuracy'),
    'agentic-rag':          ('metrics.route_accuracy',                     'route_accuracy'),
    'long-context-rag':     ('metrics.recall@3.contextual',                'recall@3 (contextual)'),
    'graph-rag':            ('metrics.n_communities',                      '# communities'),
    'multimodal-rag':       ('metrics.recall@3.joint',                     'recall@3 (joint)'),
}
def walk(d, path):
    cur = d
    for part in path.split('.'):
        if not isinstance(cur, dict) or part not in cur:
            return None
        cur = cur[part]
    return cur

## Render the leaderboard

In [3]:
rows = []
for sub in sorted(ROOT.iterdir()):
    snap = sub / 'eval-snapshot.json'
    if not snap.is_file(): continue
    data = json.loads(snap.read_text())
    tech = data.get('technique', sub.name)
    spec = HEADLINES.get(tech)
    if not spec:
        rows.append((sub.name, tech, '(no headline metric registered)', '—'))
        continue
    path, label = spec
    val = walk(data, path)
    rows.append((sub.name, tech, label, val))

print('| leaf | technique | headline metric | value |')
print('| --- | --- | --- | --- |')
for leaf, tech, lab, val in rows:
    print(f'| `{leaf}` | {tech} | {lab} | {val} |')

| leaf | technique | headline metric | value |
| --- | --- | --- | --- |
| `00-naive-rag` | naive-rag | context_recall | 0.75 |
| `01-chunking-strategies` | chunking-strategies | avg chunks/doc (fixed) | 3.2 |
| `02-embedding-comparison` | embedding-comparison | recall@3 (large-512) | 0.9231 |
| `03-hybrid-search` | hybrid-search | recall@3 (hybrid RRF) | 0.9615 |
| `04-reranking` | reranking | recall@1 (LLM reranker) | 0.9615 |
| `05-query-transformation` | query-transformation | recall@3 (HyDE) | 0.7273 |
| `06-self-rag` | self-rag | refusal_rate (unanswerable) | 1.0 |
| `07-corrective-rag` | corrective-rag | verdict_accuracy | 1.0 |
| `08-agentic-rag` | agentic-rag | route_accuracy | 1.0 |
| `09-graph-rag` | graph-rag | # communities | 6 |
| `10-multimodal-rag` | multimodal-rag | recall@3 (joint) | 0.8462 |
| `11-long-context-rag` | long-context-rag | recall@3 (contextual) | 0.8667 |
| `12-comparison-bench` | comparison-bench | (no headline metric registered) | — |


## What you can extend

* Add a column for cost/latency once `shared/llm.py` starts tracking it.
* Auto-link the leaf name to its README in the rendered Markdown.
* Diff against `main`'s snapshots and flag regressions in-notebook (CI already does this).